## U-Net Evaluations

In [19]:
import os
import numpy as np
from PIL import Image
from sklearn.metrics import precision_score, recall_score, f1_score, jaccard_score
import cv2
import re
# import os
import glob
# import re
# import numpy as np
# from sklearn.metrics import jaccard_score, precision_score, recall_score, f1_score
import tifffile as tiff
# import os
# import numpy as np
# from sklearn.metrics import jaccard_score, precision_score, recall_score, f1_score
# import rasterio

import json

from sklearn.metrics import recall_score

# with open('config.json') as json_file:
#     config = json.load(json_file)
#     corruption = config['corruption_level']


def load_image(filepath):
    # Load the image using PIL
    image = np.array(Image.open(filepath).convert('L'))
    
    # Apply Otsu's thresholding
    t, binary_image = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Convert to binary (0 and 1)
    binary_image = (binary_image > 0).astype(np.uint8)
    return binary_image

def dice_coefficient(y_true, y_pred):
    intersection = np.sum(y_true * y_pred)
    return (2. * intersection) / (np.sum(y_true) + np.sum(y_pred))

def pixel_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def specificity(y_true, y_pred):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return tn / (tn + fp)


def dice_coefficient(y_true, y_pred, smooth=1):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)

def pixel_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def specificity(y_true, y_pred):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return tn / (tn + fp)

# Paths to the folders containing the predictions and ground truths


In [20]:
corruptions = [2,8,15]
configs = ['mixed', 'erosion', 'dilation']
model_types = ['Bayesian_ResNetASPP']

parent = 'Bayesian_UNet' # only for UNet based models

# Directories for ground truth and prediction masks (Unet)
gt_folder = './GEE_Masks/GEE_resized/test_gee/'
# Directories for ground truth masks (DeepLabV3, Segformer, Maskformer)
# gt_folder = './sar_images/masks/test'

for model_type in model_types:
    
    # for the morphological noise evaluation
    for config in configs:
        for corruption in corruptions:
            
            # For ResNetASPP 
            pred_folder = f'./GEE_Output/{model_type}/{config}/{corruption}/Entropy'
            # For UNet based models
            # pred_folder = f'./GEE_Output/{parent}/{model_type}/{config}/{corruption}/Entropy'
            pred_files = glob.glob(os.path.join(pred_folder, '*.tif'))

            print(pred_folder)

            # Initialize metric lists
            dice_scores = []
            iou_scores = []
            pixel_accuracies = []
            precisions = []
            recalls = []
            f1_scores = []
            specificities = []

            # Get all prediction files
            pred_files = glob.glob(os.path.join(pred_folder, '*.tif'))

            for pred_file in pred_files:
                # Extract the number 'i' from the prediction filename
                # For UNet
                match = re.search(r'(\d+).tif', os.path.basename(pred_file))
                
                if match:
                    i = match.group(1)
                    
                    # For UNet
                    gt_filename = f'NDWI_Mask_{i}_resized.tif'
                    gt_path = os.path.join(gt_folder, gt_filename)
                    
                    # Check if the ground truth file exists
                    if os.path.exists(gt_path):
                        # Load the prediction and ground truth images
                        pred_image = load_image(pred_file)
                        gt_image = load_image(gt_path)
                        # print(np.max(pred_image))
                        # print(np.min(pred_image))
                        # Flatten the images
                        pred_flat = pred_image.flatten()
                        gt_flat = gt_image.flatten()
                        
                        # Calculate metrics
                        dice = dice_coefficient(gt_flat, pred_flat)
                        iou = jaccard_score(gt_flat, pred_flat, average='macro')
                        accuracy = pixel_accuracy(gt_flat, pred_flat)
                        precision = precision_score(gt_flat, pred_flat, average='macro', zero_division=0)
                        recall = recall_score(gt_flat, pred_flat, average='macro', zero_division=0)
                        f1 = f1_score(gt_flat, pred_flat, average='macro')
                        spec = specificity(gt_flat, pred_flat)
                        
                        # Store metrics
                        dice_scores.append(dice)
                        iou_scores.append(iou)
                        pixel_accuracies.append(accuracy)
                        precisions.append(precision)
                        recalls.append(recall)
                        f1_scores.append(f1)
                        specificities.append(spec)

                    else:
                        print(f"Ground truth file {gt_path} not found for prediction file {pred_file}.")

            # Calculate and print average metrics
            average_dice = np.mean(dice_scores)
            average_iou = np.mean(iou_scores)
            average_accuracy = np.mean(pixel_accuracies)
            average_precision = np.mean(precisions)
            average_recall = np.mean(recalls)
            average_f1 = np.mean(f1_scores)
            average_specificity = np.mean(specificities)
            
            print("Average metrics:")
            print(f"  Average Dice Coefficient: {average_dice:.4f}")
            print(f"  Average IoU: {average_iou:.4f}")
            print(f"  Average Precision: {average_precision:.4f}")
            print(f"  Average Recall: {average_recall:.4f}")
            print(f"  Average F1 Score: {average_f1:.4f}")
            print(f"  Average Specificity: {average_specificity:.4f}")
            print(f"Model's Overall Accuracy: {average_accuracy:.4f}")
    
    
    # for the gaussian noise evaluation
    
    # For UNet based models
    # pred_folder = f'./GEE_Output/{parent}/{model_type}/Gaussian/Entropy'
    # For ResnetASPP
    pred_folder = f'./GEE_Output/{model_type}/Gaussian/Entropy'
    pred_files = glob.glob(os.path.join(pred_folder, '*.tif'))

    print(pred_folder)

    # Initialize metric lists
    dice_scores = []
    iou_scores = []
    pixel_accuracies = []
    precisions = []
    recalls = []
    f1_scores = []
    specificities = []

    # Get all prediction files
    pred_files = glob.glob(os.path.join(pred_folder, '*.tif'))

    for pred_file in pred_files:
        # Extract the number 'i' from the prediction filename
        # For UNet
        match = re.search(r'(\d+).tif', os.path.basename(pred_file))
        
        if match:
            i = match.group(1)
            
            # For UNet
            gt_filename = f'NDWI_Mask_{i}_resized.tif'
            gt_path = os.path.join(gt_folder, gt_filename)
            
            # Check if the ground truth file exists
            if os.path.exists(gt_path):
                # Load the prediction and ground truth images
                pred_image = load_image(pred_file)
                gt_image = load_image(gt_path)
                # print(np.max(pred_image))
                # print(np.min(pred_image))
                # Flatten the images
                pred_flat = pred_image.flatten()
                gt_flat = gt_image.flatten()
                
                # Calculate metrics
                dice = dice_coefficient(gt_flat, pred_flat)
                iou = jaccard_score(gt_flat, pred_flat, average='macro')
                accuracy = pixel_accuracy(gt_flat, pred_flat)
                precision = precision_score(gt_flat, pred_flat, average='macro', zero_division=0)
                recall = recall_score(gt_flat, pred_flat, average='macro', zero_division=0)
                f1 = f1_score(gt_flat, pred_flat, average='macro')
                spec = specificity(gt_flat, pred_flat)
                
                # Store metrics
                dice_scores.append(dice)
                iou_scores.append(iou)
                pixel_accuracies.append(accuracy)
                precisions.append(precision)
                recalls.append(recall)
                f1_scores.append(f1)
                specificities.append(spec)

            else:
                print(f"Ground truth file {gt_path} not found for prediction file {pred_file}.")

    # Calculate and print average metrics
    average_dice = np.mean(dice_scores)
    average_iou = np.mean(iou_scores)
    average_accuracy = np.mean(pixel_accuracies)
    average_precision = np.mean(precisions)
    average_recall = np.mean(recalls)
    average_f1 = np.mean(f1_scores)
    average_specificity = np.mean(specificities)
    
    print("Average metrics:")
    print(f"  Average Dice Coefficient: {average_dice:.4f}")
    print(f"  Average IoU: {average_iou:.4f}")
    print(f"  Average Precision: {average_precision:.4f}")
    print(f"  Average Recall: {average_recall:.4f}")
    print(f"  Average F1 Score: {average_f1:.4f}")
    print(f"  Average Specificity: {average_specificity:.4f}")
    print(f"Model's Overall Accuracy: {average_accuracy:.4f}")

./GEE_Output/Bayesian_ResNetASPP/mixed/2/Entropy
Average metrics:
  Average Dice Coefficient: 0.6499
  Average IoU: 0.7300
  Average Precision: 0.8079
  Average Recall: 0.8391
  Average F1 Score: 0.7906
  Average Specificity: 0.9435
Model's Overall Accuracy: 0.9031
./GEE_Output/Bayesian_ResNetASPP/mixed/8/Entropy
Average metrics:
  Average Dice Coefficient: 0.6113
  Average IoU: 0.7007
  Average Precision: 0.7938
  Average Recall: 0.8151
  Average F1 Score: 0.7682
  Average Specificity: 0.9605
Model's Overall Accuracy: 0.8951
./GEE_Output/Bayesian_ResNetASPP/mixed/15/Entropy
Average metrics:
  Average Dice Coefficient: 0.6031
  Average IoU: 0.6877
  Average Precision: 0.7687
  Average Recall: 0.8269
  Average F1 Score: 0.7602
  Average Specificity: 0.9237
Model's Overall Accuracy: 0.8828
./GEE_Output/Bayesian_ResNetASPP/erosion/2/Entropy
Average metrics:
  Average Dice Coefficient: 0.6509
  Average IoU: 0.7321
  Average Precision: 0.8156
  Average Recall: 0.8297
  Average F1 Score: 0.7